# 02 · structures — load & prep TCR / TCR-pMHC

`pip install "kinapse[structures]"`

The `structures` module is the basis: load a PDB, fix duplicate residue numbers,
IMGT-renumber (ANARCI/ANARCII), pair α/β (or γ/δ) chains by interface contacts, and
slice to CDR/framework regions.

In [ ]:
import os
os.environ.setdefault("ANARCI_CPU", "1")   # number on CPU (safe without a GPU)

from kinapse.structures import load_tcr

tcr = load_tcr("data/example_tcr.pdb", legacy_anarci=False)
print("pairs:", len(tcr.pairs))
pair = tcr.pairs[0]

## Pairing & numbering
Each `TCRPairView` knows the **original** PDB chain ids of its α/β domains (found by
IMGT-numbering every chain and pairing them by interface contacts).

In [ ]:
print("alpha chain:", pair.alpha_chain_id, " beta chain:", pair.beta_chain_id)
pair.cdr_fr_sequences()          # per-region one-letter sequences (a method, not a property)

## The variable domain (Fv) and per-region subsets

In [ ]:
from kinapse.structures import write_pdb

var = pair.variable_structure                 # Biopython structure of the Fv
write_pdb("out_variable.pdb", var)
print("wrote out_variable.pdb")

cdr3 = pair.domain_subset(["A_CDR3"])         # just the α CDR3 loop
print("A_CDR3 atoms:", sum(1 for _ in cdr3.get_atoms()))

## Region definitions (IMGT)

In [ ]:
from kinapse import CDR_FR_RANGES
CDR_FR_RANGES

## pMHC / TCR-pMHC complexes (scaffold)
`pmhc.py` is API-ready; the current logic is TCR-focused. For a complex PDB, `TCRpMHC`
loads the TCR half with the machinery above and classifies the rest as pMHC.

In [ ]:
from kinapse.structures import TCRpMHC, load_pmhc   # noqa: F401
# complex = TCRpMHC("data/your_complex.pdb")        # when you have a TCR-pMHC complex
print("pMHC loaders available:", TCRpMHC.__name__, "/", load_pmhc.__name__)

Attach an MD trajectory to a pair (`pair.traj` → a `TrajectoryView`) — see **04 · dynamics_analysis**.